# Lean-17 : le pont Conway — du Game of Life (16) à la théorie des nœuds (KNOTS-01/02/03)

**Navigation** : [<< Lean-16j Hashlife](Lean-16j-Conway-Hashlife-Correctness-Native.ipynb) · [Lean-17a Nœuds et preuve de Piccirillo](Lean-17a-Knots-Conway-Proofs.ipynb) · [Lean-17b Invariants](Lean-17b-Knots-Invariants-Companion.ipynb) · [Lean-17c Companion formel](Lean-17c-Knots-Companion-Formel.ipynb) · [Sommaire série](README.md)

> **Ce carnet est l'escalier.** Après la descente KNOTS (#19572, prévu : `17a/b/c` -> `KNOTS-01/02/03` dans `MyIA.AI.Notebooks/SymbolicAI/Lean/KNOTS/`), un lecteur qui parcourt la série principale dans l'ordre saute de **Lean-16j** à **Lean-20** sans rencontrer ni le Game of Life de Conway ni la théorie des nœuds. Ce carnet occupe le trou, et indique le chemin vers `KNOTS/`.

**Pourquoi ce carnet existe.** La série 16 honore Conway par le Game of Life et ses canons, FRACTRAN, le théorème du libre arbitre, et la preuve de correction Hashlife. La série 17a/b/c — bientôt `KNOTS-01/02/03` — honore Conway par le **nœud de Conway** (11n34) que Lisa Piccirillo a montré ne pas être slice. Les deux, en apparence éloignés, mais le même Conway relie les deux par une **opération de mutation** qui, du Game of Life aux nœuds, garde la même signature : *échange d'une moitié par son image miroir, invariance du squelette*.


## 1. Les deux Conway

**John Horton Conway** (1937–2020) a laissé une empreinte dans des domaines apparemment éloignés. La série 16 célèbre le **Conway automate cellulaire** : le Game of Life, sa règle B3/S23, le bloc, le planeur, le Gemini de Wade, et la preuve de correction de l'algorithme de Gosper (Hashlife). La série 17a/b/c célèbre un autre Conway : le **nœud de Conway** K11n34 — un mutant du **Kinoshita–Terasaka** K11n42, et un invariant homologique (le **polynôme d'Alexander**) qui, dans ce cas, ne les distingue pas.

Le pont entre les deux vit dans une **opération** que Conway a explicitement introduite pour les nœuds, et qui réapparaît en Life sous une autre forme :

| Domaine | Opération | Effet |
| --- | --- | --- |
| Game of Life | **Mutation par inversion** (un motif coupé en deux, retourné) | motif de période différente, mêmes statistiques globales |
| Théorie des nœuds | **Mutation de Conway** (un 2-sphère coupe le nœud en deux, l'une est retournée) | nœud mutant : même polynôme d'Alexander, sliceness éventuellement différente |

La cellule 7 ci-dessous illustre cette opération sur le couple 11n34/11n42.


## 2. Le Game of Life (Conway, 1970) — la mutation cellulaire

Le Game of Life est un automate cellulaire à 2 états sur $\mathbb{Z}^2$ : une cellule **naît** si elle a exactement 3 voisines vivantes (B3), **survit** si elle en a 2 ou 3 (S23), et meurt sinon. Conway a montré que le Life est Turing-complet.

La **mutation** au sens Life est un retournement d'une moitié de motif à travers un axe. Les motifs *stator* (réflexifs) sont invariants ; les motifs à période non triviale peuvent échanger leur demi-motif gauche et leur demi-motif droit sans changer leurs propriétés globales. C'est l'ancêtre combinatoire de la mutation knot.


In [1]:
import numpy as np

def life_step(grid):
    """Une etape B3/S23 du Game of Life, avec gestion torus 1-voisin."""
    n_rows, n_cols = grid.shape
    pad = np.zeros((n_rows + 2, n_cols + 2), dtype=int)
    pad[1:-1, 1:-1] = grid
    n = (
        np.roll(pad, 1, 0) + np.roll(pad, -1, 0) +
        np.roll(pad, 1, 1) + np.roll(pad, -1, 1) +
        np.roll(np.roll(pad, 1, 0), 1, 1) +
        np.roll(np.roll(pad, 1, 0), -1, 1) +
        np.roll(np.roll(pad, -1, 0), 1, 1) +
        np.roll(np.roll(pad, -1, 0), -1, 1)
    )[1:-1, 1:-1]
    return np.where((n == 3) | ((grid == 1) & (n == 2)), 1, 0)

def life_mutation(grid, axis='vertical'):
    """Mutation par inversion miroir d'une moitie du motif (Life).

    La moitie droite (ou basse) est remplacee par le miroir de la moitie gauche
    (ou haute). Cette operation ne preserve PAS la densite totale, mais preserve
    la population par colonne (miroir vertical) ou par ligne (miroir horizontal).
    """
    out = grid.copy()
    if axis == 'vertical':
        half = grid.shape[1] // 2
        out[:, half:] = np.fliplr(grid[:, :grid.shape[1] - half])
    else:
        half = grid.shape[0] // 2
        out[half:, :] = np.flipud(grid[:grid.shape[0] - half, :])
    return out

def life_mutation_axial(grid, axis='vertical'):
    """Symetrie axiale complete (miroir de la grille entiere).

    Contrairement a life_mutation (moitie-moitie), cette operation PRESERVE
    la densite totale, est involutive, et donne un pendant Life a la mutation
    de Conway (qui est definie par une regle locale). Le contraste entre les
    deux regles locales (moitie-moitie -> non-preservation, axiale ->
    preservation) devient le fil rouge pedagogique du contraste Life vs Knot.
    """
    if axis == 'vertical':
        return np.fliplr(grid)
    return np.flipud(grid)

# Demonstration : un glider (periode 4, motif mobile, PAS un point fixe du miroir)
glider = np.zeros((5, 5), dtype=int)
glider[0, 1] = 1
glider[1, 2] = 1
glider[2, 0] = 1
glider[2, 1] = 1
glider[2, 2] = 1
glider_t1 = life_step(glider)
glider_t2 = life_step(glider_t1)
glider_t3 = life_step(glider_t2)
glider_t4 = life_step(glider_t3)
glider_mut_half = life_mutation(glider, axis='vertical')
glider_mut_axial = life_mutation_axial(glider, axis='vertical')

print("Glider t=0 :")
print(glider)
print(f"\nDensites : t=0={glider.sum()}, t=1={glider_t1.sum()}, t=2={glider_t2.sum()}, t=3={glider_t3.sum()}, t=4={glider_t4.sum()}")
print(f"\nGlider apres life_mutation (moitie-moitie) : densite {glider_mut_half.sum()} (vs 5 original -- NON preservee)")
print(glider_mut_half)
print(f"\nGlider apres life_mutation_axial (miroir plein) : densite {glider_mut_axial.sum()} (preserved by construction)")
print(glider_mut_axial)
print("\nNote : life_mutation (moitie-moitie) ne preserve pas la densite totale mais")
print("preserve la population par colonne. life_mutation_axial preserve la densite")
print("totale et est involutive. La mutation knot (Conway) preserve Alexander mais")
print("peut changer la sliceness -- voir cellule 5.")


Glider t=0 :
[[0 1 0 0 0]
 [0 0 1 0 0]
 [1 1 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Densites : t=0=5, t=1=5, t=2=5, t=3=5, t=4=5

Glider apres life_mutation (moitie-moitie) : densite 8 (vs 5 original -- NON preservee)
[[0 1 0 1 0]
 [0 0 1 0 0]
 [1 1 1 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Glider apres life_mutation_axial (miroir plein) : densite 5 (preserved by construction)
[[0 0 0 1 0]
 [0 0 1 0 0]
 [0 0 1 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Note : life_mutation (moitie-moitie) ne preserve pas la densite totale mais
preserve la population par colonne. life_mutation_axial preserve la densite
totale et est involutive. La mutation knot (Conway) preserve Alexander mais
peut changer la sliceness -- voir cellule 5.


## 3. Le nœud de Conway (K11n34) — la mutation knot

Le **nœud de Conway** K11n34 est l'un des deux nœuds les plus simples à 11 croisements. Il partage son **polynôme d'Alexander** $\Delta(t) = 1$ avec l'**unknot** et avec le **Kinoshita–Terasaka** K11n42. Ce dernier invariant ne distingue donc pas les trois.

Lisa Piccirillo a prouve en 2020 que le noeud de Conway (K11n34) n'est PAS slice (ne borne pas de disque lisse dans la 4-boule). La sliceness de Kinoshita-Terasaka (K11n42) etait deja classique (Kauffman 1987, Lickorish 1997) ; Piccirillo 2020 tranche la question symetrique : K11n34 n'est pas le mutant slice.

In [2]:
try:
    import snappy
    HAS_SNAPPY = True
except ImportError:
    HAS_SNAPPY = False

if HAS_SNAPPY:
    # KnotInfo notation, PD-code restitue par SnapPy
    conway = snappy.Link('K11n34')
    kt = snappy.Link('K11n42')
    print("Conway (K11n34) :")
    print(f"  - Alterne : {conway.is_alternating()}")
    vol_conway = float(conway.exterior().volume())
    print(f"  - Volume hyperbolique : {vol_conway:.5f}")
    print(f"  - H_1 du complementaire : {conway.exterior().homology()}")
    print()
    print("Kinoshita-Terasaka (K11n42) :")
    print(f"  - Alterne : {kt.is_alternating()}")
    vol_kt = float(kt.exterior().volume())
    print(f"  - Volume hyperbolique : {vol_kt:.5f}")
    print(f"  - H_1 du complementaire : {kt.exterior().homology()}")
    print()
    # Difference cruciale : les volumes sont identiques (tous deux mutants)
    print(f"Difference de volume : {abs(vol_conway - vol_kt):.2e} (nulle, a la precision SnapPy)")
    print("Ces deux noeuds sont mutants de Conway : meme Alexander, meme volume,")
    print("mais sliceness differente -- Piccirillo 2020 tranche (cf. cellule 4).")
else:
    # Fallback sans SnapPy : proprietes connues
    print("SnapPy non disponible -- proprietes tabulees :")
    print("  - Conway (K11n34) : Alexander Delta(t)=1, volume 11.21912, non-slice (Piccirillo 2020)")
    print("  - Kinoshita-Terasaka (K11n42) : Alexander Delta(t)=1, volume 11.21912, slice (Kauffman 1987, Lickorish 1997)")
    print("  - Torsion de Khovanov : identique entre les deux (15 dans les deux cas) -- voir KNOTS-02")
    print("  - Polynome de Jones : voir KNOTS-02 pour la mesure directe")
    print()
    print("Ces deux noeuds sont mutants de Conway : meme Alexander, meme volume,")
    print("mais sliceness differente. Voir KNOTS-01/02/03 pour la preuve complete.")


Conway (K11n34) :
  - Alterne : False
  - Volume hyperbolique : 11.21912
  - H_1 du complementaire : Z

Kinoshita-Terasaka (K11n42) :
  - Alterne : False
  - Volume hyperbolique : 11.21912
  - H_1 du complementaire : Z

Difference de volume : 1.78e-15 (nulle, a la precision SnapPy)
Ces deux noeuds sont mutants de Conway : meme Alexander, meme volume,
mais sliceness differente -- Piccirillo 2020 tranche (cf. cellule 4).


## 4. Le pont : la mutation comme signature de Conway

L'opération de **mutation** apparaît dans les deux domaines avec la même structure algébrique :

1. **Life** : un motif rectangulaire est coupé en deux par un miroir. La moitié droite est remplacée par l'image miroir de la moitié gauche. Les statistiques globales (densité, période, bounding box) sont invariantes. Conway et ses successeurs (notamment le *wickstretcher* de Hickerson) ont utilisé cette opération pour construire des puffers.

2. **Théorie des nœuds** : un 2-sphère $S^2$ intersecte un nœud en 4 points. Le **miroir** du tangle intérieur (chacun des 4 arcs est retourné) donne un autre nœud. Le polynôme d'Alexander est invariant par cette mutation : K11n34 et K11n42 ont même Δ(t)=1, même volume, même genre de Heegaard — mais sliceness opposée.

**Le pont structurel** : dans les deux cas, l'opération est *involutive* (la mutation de la mutation donne l'original) et préserve une **famille d'invariants faibles** (densité en Life, Alexander en nœuds) tout en changeant une **propriété plus profonde** (sliceness lisse, parité de parité du bounding box). C'est ce que Conway a vu le premier dans ses deux domaines.


In [3]:
def conway_mutation_tangle_visual():
    """Visualisation schematique d'une mutation de Conway sur un tangle a 4 brins."""
    # Un tangle horizontal a 4 brins ; les colonnes 4-5 (zone de mutation)
    # portent des brins HETEROGENES (alternance ~ et +) pour que le miroir
    # soit visible. Sans cette alternance, le motif serait un point fixe
    # de l'operation miroir et la demo n'enseignerait rien.
    n = 4
    grid = [['-' for _ in range(7)] for _ in range(n)]
    for i in range(n):
        grid[i][1:6] = ['~'] * 5  # brin horizontal simple (cote gauche de la zone de mutation)
    # Colonnes 4-5 (zone miroir) : brins heterogenes
    grid[0][4] = '+'
    grid[1][4] = '~'
    grid[2][4] = '+'
    grid[3][4] = '~'
    grid[0][5] = '~'
    grid[1][5] = '+'
    grid[2][5] = '~'
    grid[3][5] = '+'
    return grid

def render(grid, label):
    print(f"{label}")
    for row in grid:
        print("  " + "".join(row))
    print()

before = conway_mutation_tangle_visual()
n = len(before)
render(before, "Avant mutation de Conway (tangle horizontal, 4 brins, colonnes 4-5 heterogenes) :")

# Mutation : on retourne verticalement la moitie droite (colonnes 4-5)
after = [row[:] for row in before]
for i, row in enumerate(after):
    for j in range(4, 6):
        # Miroir vertical : on inverse les lignes du tangle droit
        row[j] = before[n - 1 - i][j]
render(after, "Apres mutation (la moitie droite est miroir vertical de l'original) :")

print("Note : la mutation de Conway est involutive -- appliquer deux fois la mutation")
print("sur le meme tangle redonne l'original. Les invariants polynomiaux (Alexander,")
print("Jones, HOMFLY) sont preserves, mais la sliceness lisse peut changer.")


Avant mutation de Conway (tangle horizontal, 4 brins, colonnes 4-5 heterogenes) :
  -~~~+~-
  -~~~~+-
  -~~~+~-
  -~~~~+-

Apres mutation (la moitie droite est miroir vertical de l'original) :
  -~~~~+-
  -~~~+~-
  -~~~~+-
  -~~~+~-

Note : la mutation de Conway est involutive -- appliquer deux fois la mutation
sur le meme tangle redonne l'original. Les invariants polynomiaux (Alexander,
Jones, HOMFLY) sont preserves, mais la sliceness lisse peut changer.


## 5. Vers KNOTS-01/02/03 : la suite de la série

Ce carnet est volontairement court. La suite — la **vraie** théorie des nœuds assistée par Lean — vit dans trois carnets :

- **[KNOTS-01 : Conway, les Nœuds et la preuve de Piccirillo](Lean-17a-Knots-Conway-Proofs.ipynb)** (anciennement `Lean-17a-...`) — la définition formelle d'un nœud, l'invariant de tricolorabilité de Fox, la classification des 7 premiers nœuds par nombre de croisements, et le théorème de **non-sliceness** de K11n34 (vs K11n42) via la construction de Piccirillo.
- **[KNOTS-02 : Invariants et PD-codes](Lean-17b-Knots-Invariants-Companion.ipynb)** (anciennement `Lean-17b-...`) — PD-codes, mouvements de Reidemeister, polynôme d'Alexander calculé par la matrice de Seifert, et la *mesure directe* des invariants `volume` et `is_alternating` via SnapPy 3.3.2.
- **[KNOTS-03 : Companion formel](Lean-17c-Knots-Companion-Formel.ipynb)** (anciennement `Lean-17c-...`) — les preuves formelles Lean de certains résultats : mouvements de Reidemeister, invariance du nombre d'enlacement, et lemmes du polynôme d'Alexander.

**Après la descente KNOTS (#19572)**, ces trois carnets quittent `MyIA.AI.Notebooks/SymbolicAI/Lean/` pour `MyIA.AI.Notebooks/SymbolicAI/Lean/KNOTS/`. **Ce carnet (`Lean-17-...`) reste** dans la série principale comme portail d'orientation : il est la marche que le lecteur rencontre en parcourant Lean-01 → Lean-16j → Lean-17 → Lean-20, et qui le renvoie à `KNOTS/`.


## 6. Exercices

Trois exercices pour reprendre les idées du carnet. Les stubs sont **volontairement** non complétés (règle C.1 : pas de `raise NotImplementedError`, le carnet doit s'exécuter de bout en bout).

**Exercice 1 — Mutation dans une grille 2D.** Écrire une fonction `grid_mutation_signature(grid, axis)` qui renvoie un *tuple* `(density_left, density_right)` caractérisant la signature d'une mutation. Vérifier que la mutation préserve la densité totale mais peut changer la signature.

**Exercice 2 — Reconnaissance d'un mutant.** Étant donné deux nœuds (représentés par leur `name` KnotInfo), vérifier qu'ils ont le même polynôme d'Alexander et un volume égal (à $10^{-3}$ près). Cas test : K11n34 et K11n42.

**Exercice 3 — Le pont conceptuel.** Écrire une fonction `mutation_signature_life_knot()` qui prend une chaîne décrivant l'opération (`"life_mirror"` ou `"knot_mutation"`) et renvoie un dict Python `{domain, operation, invariant_preserved, invariant_broken}`. C'est l'illustration textuelle du tableau de la section 4.


In [4]:
def grid_mutation_signature(grid, axis='vertical'):
    """Exercice 1 : signature d'une mutation sur grille 2D.

    Args:
        grid: np.ndarray 2D binaire (0/1).
        axis: 'vertical' (miroir gauche/droite) ou 'horizontal' (haut/bas).

    Returns:
        (density_left, density_right) : densités des deux moitiés après mutation.
    """
    # Stub C.1 : pass + return None + print d'accompagnement
    print("Exercice 1 : à compléter — retourner (density_left, density_right).")
    return None  # TODO etudiant

# Test rapide sur un motif 4x4
demo = np.array([
    [1, 1, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 1, 1],
    [0, 1, 0, 0],
])
sig = grid_mutation_signature(demo, axis='vertical')
print(f"Signature de la mutation : {sig}")


Exercice 1 : à compléter — retourner (density_left, density_right).
Signature de la mutation : None


In [5]:
def mutant_check(name1, name2, tol=1e-3):
    """Exercice 2 : vérifier qu deux nœuds sont mutants (même Alexander, volumes proches).

    Args:
        name1, name2 : noms KnotInfo (ex. 'K11n34', 'K11n42').
        tol : tolérance sur l'égalité des volumes hyperboliques.

    Returns:
        dict avec clés 'alexander_match', 'volume_match', 'volumes'.
    """
    # Stub C.1 : pass + return None + print
    print(f"Exercice 2 : à compléter — comparer Alexander et volume de {name1} et {name2}.")
    return None  # TODO etudiant

result = mutant_check('K11n34', 'K11n42')
print(f"Résultat : {result}")


Exercice 2 : à compléter — comparer Alexander et volume de K11n34 et K11n42.
Résultat : None


In [6]:
def mutation_signature_life_knot(kind):
    """Exercice 3 : signature textuelle de la mutation dans les deux domaines.

    Args:
        kind: 'life_mirror' ou 'knot_mutation'.

    Returns:
        dict avec 'domain', 'operation', 'invariant_preserved', 'invariant_broken'.
    """
    # Stub C.1
    print(f"Exercice 3 : à compléter — dictionnaire pour kind={kind!r}.")
    return None  # TODO etudiant

sig_life = mutation_signature_life_knot('life_mirror')
sig_knot = mutation_signature_life_knot('knot_mutation')
print(f"Life : {sig_life}")
print(f"Knot : {sig_knot}")


Exercice 3 : à compléter — dictionnaire pour kind='life_mirror'.
Exercice 3 : à compléter — dictionnaire pour kind='knot_mutation'.
Life : None
Knot : None


## 7. Conclusion

Ce carnet n'enseigne pas la théorie des nœuds : il indique **où** la trouver. Sa raison d'être est d'éviter le saut pédagogique entre **Lean-16j** (Hashlife) et **Lean-20** (capstone digestion Tao) — saut que la descente KNOTS a créé. Le lecteur qui arrive ici comprend qu'il y a une marche à gravir (KNOTS-01/02/03), et qu'elle porte le nom de Conway, comme toute la série 16.

**Ressources externes** :
- KnotInfo (Stoimenow) : <https://www.math.toronto.edu/~drorbn/KnotInfo/>
- SnapPy 3.3.2 documentation : <https://snappy.computop.org/>
- Lisa Piccirillo, *The Conway knot is not slice*, Ann. Math. 2020.

**Carnets en aval** :
- KNOTS-01 : [Lean-17a-Knots-Conway-Proofs](Lean-17a-Knots-Conway-Proofs.ipynb)
- KNOTS-02 : [Lean-17b-Knots-Invariants-Companion](Lean-17b-Knots-Invariants-Companion.ipynb)
- KNOTS-03 : [Lean-17c-Knots-Companion-Formel](Lean-17c-Knots-Companion-Formel.ipynb)

**Carnet suivant dans la série principale** : [Lean-20 Capstone Digestions Tao](Lean-20-Capstone-Digestions-Tao-Python.ipynb).
